# CMR

## Temporal Context

In [36]:
# | export

from jax import jit
import jax.numpy as jnp
from plum import dispatch
from jaxtyping import Shaped, Integer, Float, Array, Scalar
from flax import struct

class TemporalContext(struct.PyTreeNode):
    state: Float[Array, "context_units"]
    start_context_input: Float[Array, "context_units"]
    delay_context_input: Float[Array, "context_units"]

def initialize_temporal_context(item_count: int | Shaped[Integer, ""]) -> TemporalContext:
    "Initialize a temporal context representation."
    context_state = jnp.zeros(item_count + 2)
    return TemporalContext(
        state=context_state.at[0].set(1),
        start_context_input=context_state.at[0].set(1),
        delay_context_input=context_state.at[-1].set(1),
    )

@dispatch
def integrate(
    context_state: Float[Array, "context_units"],
    context_input: Float[Array, "context_units"],
    drift_rate: float | Float[Array, ""]
    ) -> Float[Array, "context_units"]:
    "Integrate an input representation into a contextual state."
    rho = jnp.sqrt(
        1 + jnp.square(drift_rate) * (jnp.square(context_state * context_input) - 1)
    ) - (drift_rate * (context_state * context_input))

    return (rho * context_state) + (drift_rate * context_input)

@dispatch
def integrate(
    context: TemporalContext,
    context_input: Float[Array, "context_units"],
    drift_rate: float | Float[Array, ""]
    ) -> TemporalContext:
    "Integrate an input representation into a temporal context representation."
    return context.replace(state=integrate(context.state, context_input, drift_rate))

In [47]:
def test_integrate_second_context_unit():

    def f():
        item_count = 10
        encoding_drift_rate = 0.3
        context = initialize_temporal_context(item_count)

        context_input = jnp.zeros(item_count + 2)
        context_input = context_input.at[1].set(1)
        context_input = context_input / jnp.sqrt(jnp.sum(jnp.square(context_input)))

        return integrate(context, context_input, encoding_drift_rate)

    desired_result = jnp.array([0.9539392, 0.3, 0. , 0. , 0., 0.,
    0. , 0. , 0. , 0.       , 0.       , 0.       ])
    assert jnp.allclose(f().state, desired_result)
    assert jnp.allclose(jit(f)().state, desired_result)

test_integrate_second_context_unit()

## Linear Associative $M^{FC}$
In the Context Maintenance and Retrieval (CMR) model specifications, $M^{FC}$ represents forward-going connections between the feature layer and the context layer. 
It is represented as a matrix of weights, where each row corresponds to a feature unit, and each column corresponds to a context unit.

In the original specification of CMR, $M^{FC}$ is initialized as a diagonal matrix.
The diagonal elements of Mfc are set to 1 minus a `learning_rate` parameter ranging between 0 and 1. 
Off-diagonal elements are initialized to 0.
This corresponds to a pattern of pre-experimental item-to-context associations in which each item is associated with an orthogonal (unique) contextual feature based on the value of the `learning_rate` parameter.
Even though it is referred to as our `learning_rate,` the parameter has this role because its broader purpose is to control the ratio of the influence of pre-experimental item-to-context associations to post-experimental item-to-context associations.

During encoding, the outer product of the feature vector and the context vector is added to $M^{FC}$, enacting a Hebbian learning rule.
The `learning_rate` parameter scales the outer product added to $M^{FC}$.
To retrieve contextual features associated with a probe feature vector (activations), the dot product of the feature vector and $M^{FC}$ is computed.

In [49]:
# | export

import jax.numpy as jnp
import jax
lb = jnp.finfo(float).eps

class Mfc(struct.PyTreeNode):
    "A feature-to-context linear associative memory"
    state: Float[Array, "feature_units context_units"]

def initialize_mfc(
        item_count: int | Shaped[Integer, ""], 
        learning_rate: float | Float[Array, ""]
        ) -> Mfc:
    "Initialize a feature-to-context associative memory"
    initial_state = jnp.eye(item_count, item_count + 2)
    initial_state = jnp.hstack([jnp.zeros((item_count, 1)), initial_state[:, :-1]])
    initial_state = initial_state * (1 - learning_rate)
    return Mfc(initial_state)

def associate(
        memory: Mfc,
        learning_rate: float | Float[Array, ""],
        item_feature_pattern: Float[Array, "feature_units"],
        context_feature_pattern: Float[Array, "context_units"],
        ) -> Mfc:
    "Associate item and context feature patterns in a feature-to-context associative memory"
    return memory.replace(state=memory.state + (
        learning_rate * jnp.outer(item_feature_pattern, context_feature_pattern)
    ))

@dispatch
def activations(
        memory_state: Float[Array, """input_units output_units"""],
        probe: Float[Array, "input_units"],
        ) -> Float[Array, "output_units"]:
    "Activation of output units given an probe and a linear associative memory"
    activation = jnp.dot(probe, memory_state)
    return activation / jnp.sqrt(jnp.sum(jnp.square(activation)) + lb)

@dispatch
def activations(
    memory: Mfc,
    item_feature_probe: Float[Array, "feature_units"],
    ) -> Float[Array, "context_units"]:
    "Activation of context feature units given an item feature probe and a feature-to-context associative memory"
    return activations(memory.state, item_feature_probe)

In [56]:
def test_associate_first_item_and_sixth_context_in_mfc():
    
    item_count = 16
    learning_rate = 0.1
    
    def f():
        mfc = initialize_mfc(item_count, learning_rate)
        mfc = associate(mfc, learning_rate, jnp.eye(item_count)[0], jnp.eye(item_count + 2)[5])
        return mfc
    
    desired_result = jnp.array([0. , 0.9, 0. , 0. , 0. , 0.1, 0. , 0. , 0. , 0. , 0. , 0. , 0. ,
        0. , 0. , 0. , 0. , 0. ])
    assert jnp.allclose(f().state[0], desired_result)
    assert jnp.allclose(jit(f)().state[0], desired_result)

test_associate_first_item_and_sixth_context_in_mfc()

## Linear Associative $M^{CF}$

In [ ]:
#| export

class Mcf(struct.PyTreeNode):
    "A context-to-feature linear associative memory"
    state: Float[Array, """context_units feature_units"""]

class LinearAssociativeMcf(Mcf):
    pass

def initialize_mcf(
        item_count: int | Shaped[Integer, ""], 
        shared_support: float | Float[Array, ""],
        item_support: float | Float[Array, ""],
        ) -> LinearAssociativeMcf:
    "Initialize a context-to-feature linear associative memory"
    mcf = jnp.full((item_count, item_count), shared_support)
    mcf = mcf.at[jnp.diag_indices(item_count)].set(item_support)
    mcf = jnp.vstack((jnp.zeros((1, item_count)), mcf, jnp.zeros((1, item_count))))
    return Mcf(mcf)


def associate(memory: jax.Array, learning_rate: float, pattern_A: jax.Array, pattern_B: jax.Array) -> jax.Array:
    return memory + (learning_rate * jnp.outer(pattern_A, pattern_B))


def activations(memory: jax.Array, context_feature_probe: jax.Array, choice_sensitivity: float):
    a = jnp.dot(context_feature_probe, memory)
    return scale_activation(a, choice_sensitivity)


def scale_activation(activation: jax.Array, scale: float):
    log_activation = jnp.log(activation)

    return jax.lax.cond(
        jnp.logical_and(jnp.any(activation != 0), scale != 1), # if any activation is non-zero and scale is not 1
        lambda _: jnp.exp(scale * (log_activation - jnp.max(log_activation))),
        lambda _: activation,
        None,
    )